# 의료인력 지역 불평등 추가 분석 Colab

기존 `medical_workforce_kosis_colab.ipynb` 에서 생성한 패널 데이터를 바탕으로  
**연구가설 H1·H2·H3** 를 통계적으로 검증하고 논문용 그래프 7개를 생성합니다.

| 가설 | 내용 | 주요 방법 |
|------|------|-----------|
| **H1** | 수도권·비수도권 의사밀도 격차가 시간이 갈수록 커진다 | 선형 추세 회귀, 변이계수(CV) |
| **H2** | 고령화율↑ → 의사밀도↑ 관계가 수도권에서 더 강하다 | 피어슨 상관, 집단별 비교 |
| **H3** | 인구 유출 지역일수록 의사밀도가 낮다 | 독립 t-검정, 피어슨 상관 |

---

**기본 설정값** 은 아래 저장소 기준입니다.  
다른 경로에 올렸다면 **셀 1 의 4개 변수만** 수정하세요.

```
GitHub 사용자 : runmc3812
저장소        : Healthcare_Informatics_Integration-2026-
브랜치        : main
폴더          : 6조/medical_workforce_kosis
```

**결과물** — `results/` 폴더에 자동 저장됩니다.

| 파일 | 내용 |
|------|------|
| `plot_H1_gap_trend.png` | 수도권·비수도권 추세 + 격차 이중축 그래프 |
| `plot_H1_CV_trend.png` | 변이계수(CV) 추세 |
| `plot_H2_scatter_aging_doctor.png` | 고령화율×의사밀도 산점도 (수도권 분리) |
| `plot_H2_bubble_2024.png` | 2024년 버블차트 (버블=인구 규모) |
| `plot_H3_boxplot_migration.png` | 순이동 방향별 박스플롯 |
| `plot_slope_by_region.png` | 시도별 의사밀도 증가 속도 비교 |
| `plot_quadrant_2024.png` | 2024년 고령화–의사밀도 사분면 |
| `H1_gap_table.csv` | 연도별 수도권·비수도권 격차 테이블 |
| `H1_CV_table.csv` | 연도별 변이계수 테이블 |
| `snapshot_2024.csv` | 2024년 시도별 주요 지표 |
| `slope_by_region.csv` | 시도별 의사밀도 증가 기울기 |
| `panel_data_final.csv` | 병합 완료 패널 데이터 전체 |


## 셀 1. 저장소 설정 & 분석 스크립트 다운로드

> **여기만 수정하면 됩니다.** `GITHUB_USER`, `REPO_NAME`, `BRANCH`, `BASE_PATH` 4개 값을 실제 업로드 경로에 맞게 바꾸세요.

In [ ]:
# =====================================================
# 저장소 설정 — 이 4개 값만 수정하세요
# =====================================================
GITHUB_USER = "runmc3812"
REPO_NAME   = "Healthcare_Informatics_Integration-2026-"
BRANCH      = "main"
BASE_PATH   = "6조/medical_workforce_kosis"
# =====================================================

!pip install -q requests

import os, sys, requests
from urllib.parse import quote

def gh_raw_url(*parts):
    encoded = "/".join(quote(str(p)) for p in parts)
    return f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{encoded}"

# src/ 폴더의 추가 분석 스크립트를 가져옵니다
script_url = gh_raw_url(BASE_PATH, "src", "medical_workforce_hypothesis_analysis.py")
resp = requests.get(script_url, timeout=60)
resp.raise_for_status()

with open("medical_workforce_hypothesis_analysis.py", "w", encoding="utf-8") as f:
    f.write(resp.text)

print(f"✅ 스크립트 다운로드 완료 ({len(resp.text):,} bytes)")
print(f"   URL: {script_url}")

## 셀 2. 패키지 설치 & 한글 폰트

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy
!apt-get -qq install -y fonts-nanum

import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")
os.makedirs("/content/results", exist_ok=True)

# 나눔고딕 폰트 등록
_font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(_font_path):
    fm.fontManager.addfont(_font_path)
    plt.rcParams.update({
        "font.family"       : "NanumGothic",
        "axes.unicode_minus": False,
        "figure.dpi"        : 120,
        "axes.spines.top"   : False,
        "axes.spines.right" : False,
    })
    print("✅ NanumGothic 폰트 등록 완료")
else:
    print("⚠ 폰트 없음 — 한글이 깨질 수 있습니다. 셀을 다시 실행하세요.")

## 셀 3. KOSIS CSV 다운로드 & 패널 데이터 구성

저장소 `data/` 폴더의 CSV 4개를 직접 가져와 **지역×연도 패널 형태**로 병합합니다.

| 파일 | 변수 | 연도 범위 |
|------|------|-----------|
| 인구_천명당_의료기관_종사_의사수_… | 의사밀도 | 2007~2024 |
| 고령인구비율_… | 고령인구비율 | 2007~2025 |
| 주민등록인구_… | 주민등록인구 | 2007~2025 |
| 순이동인구_… | 순이동인구 | 2007~2025 |

In [ ]:
# CSV 파일명 목록
DATA_FILES = {
    "의사밀도"   : "인구_천명당_의료기관_종사_의사수_시도_시_군_구__20260324172132.csv",
    "고령인구비율": "고령인구비율_시도_시_군_구__20260324171859.csv",
    "주민등록인구": "주민등록인구_시도_시_군_구__20260324172055.csv",
    "순이동인구"  : "순이동인구_시도_시_군_구__20260324172019.csv",
}

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

for var, fname in DATA_FILES.items():
    url  = gh_raw_url(BASE_PATH, "data", fname)
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    path = f"{DATA_DIR}/{fname}"
    with open(path, "wb") as f:
        f.write(resp.content)
    print(f"  ✔ {var:8s}  {fname}  ({len(resp.content):,} bytes)")

print("\n✅ CSV 4개 다운로드 완료")

# ── 파싱 함수 ──────────────────────────────────────
ENC = "cp949"

def _parse_wide(path):
    """KOSIS wide-format CSV → 지역×연도 (연도당 첫 번째 서브컬럼만 추출)"""
    df   = pd.read_csv(path, encoding=ENC, header=None)
    data = df.iloc[2:].reset_index(drop=True)
    years = []
    for c in df.iloc[0, 1:]:
        try:
            y = int(str(c).split(".")[0])
            years.append(y if 2000 <= y <= 2030 else None)
        except:
            years.append(None)
    records = []
    for _, row in data.iterrows():
        rec  = {"지역": row.iloc[0]}
        prev = None
        for yr, val in zip(years, row.iloc[1:]):
            if yr and yr != prev:
                rec[yr] = val
                prev = yr
        records.append(rec)
    return pd.DataFrame(records)

def _to_long(df_wide, value_name):
    years = [c for c in df_wide.columns if isinstance(c, int)]
    long  = df_wide.melt(id_vars="지역", value_vars=years,
                          var_name="연도", value_name=value_name)
    long["연도"]     = long["연도"].astype(int)
    long[value_name] = pd.to_numeric(long[value_name], errors="coerce")
    return long

# ── 로드 & 병합 ─────────────────────────────────────
doc_long = _to_long(_parse_wide(f"{DATA_DIR}/{DATA_FILES['의사밀도']}"),    "의사밀도")
age_long = _to_long(_parse_wide(f"{DATA_DIR}/{DATA_FILES['고령인구비율']}"), "고령인구비율")
pop_long = _to_long(_parse_wide(f"{DATA_DIR}/{DATA_FILES['주민등록인구']}"), "주민등록인구")

_mv_raw  = pd.read_csv(f"{DATA_DIR}/{DATA_FILES['순이동인구']}", encoding=ENC, header=None)
_mv_data = _mv_raw.iloc[2:].reset_index(drop=True)
_mv_yrs  = [int(c) for c in _mv_raw.iloc[0, 2:]]
_mv_recs = [
    {"지역": r.iloc[0], **{yr: r.iloc[j+2] for j, yr in enumerate(_mv_yrs)}}
    for _, r in _mv_data.iterrows() if str(r.iloc[1]).strip() == "계"
]
mv_long = _to_long(pd.DataFrame(_mv_recs), "순이동인구")

df = (doc_long
      .merge(age_long, on=["지역", "연도"])
      .merge(pop_long, on=["지역", "연도"])
      .merge(mv_long,  on=["지역", "연도"], how="left"))
df = df[df["지역"] != "전국"].copy()
df["주민등록인구"] = pd.to_numeric(df["주민등록인구"], errors="coerce")
df["순이동인구"]   = pd.to_numeric(df["순이동인구"],   errors="coerce")

# 파생 변수
CAPITAL       = ["서울특별시", "경기도", "인천광역시"]
df["수도권"]   = df["지역"].isin(CAPITAL).astype(int)
df["수도권명"] = df["수도권"].map({1: "수도권", 0: "비수도권"})
df["유출입"]   = df["순이동인구"].apply(
    lambda x: "순유입" if x > 0 else ("순유출" if x < 0 else "균형") if pd.notna(x) else np.nan
)

df_valid = df.dropna(subset=["의사밀도", "고령인구비율"]).copy()

print(f"\n✅ 패널 완성")
print(f"   행수  : {len(df_valid):,}")
print(f"   시도  : {df_valid['지역'].nunique()}개")
print(f"   연도  : {df_valid['연도'].min()}~{df_valid['연도'].max()}")
print()
df_valid.head(6)[[
    "지역", "연도", "의사밀도", "고령인구비율", "주민등록인구", "수도권명", "유출입"
]]

## 셀 4. H1 통계 검정 — 수도권·비수도권 격차 추세

- **격차 = 수도권 의사밀도 평균 − 비수도권 의사밀도 평균** (연도별)
- 격차가 연도에 따라 통계적으로 유의하게 증가하는지 선형 회귀로 검증
- 지역 간 불평등 전반은 **변이계수(CV = 표준편차/평균)** 로 보조 검증

In [ ]:
# ── 연도별 수도권·비수도권 평균 ─────────────────────
gap_tbl = (
    df_valid
    .groupby(["연도", "수도권명"])["의사밀도"]
    .mean()
    .unstack("수도권명")
)
gap_tbl["격차"]    = gap_tbl["수도권"] - gap_tbl["비수도권"]
gap_tbl["격차_배율"] = (gap_tbl["격차"] / gap_tbl.loc[2007, "격차"]).round(2)

# ── 선형 추세 (격차 ~ 연도) ──────────────────────────
years_arr              = gap_tbl.index.values.astype(float)
slope_h1, ic_h1, r_h1, p_h1, _ = stats.linregress(years_arr, gap_tbl["격차"].values)
r2_h1 = r_h1 ** 2

# ── 변이계수(CV) ─────────────────────────────────────
cv_tbl        = df_valid.groupby("연도")["의사밀도"].agg(["mean", "std"])
cv_tbl["CV"]  = cv_tbl["std"] / cv_tbl["mean"]
sl_cv, ic_cv, r_cv, p_cv, _ = stats.linregress(
    cv_tbl.index.values.astype(float), cv_tbl["CV"].values
)

print("=" * 58)
print("  H1 검증: 수도권·비수도권 의사밀도 격차 추세")
print("=" * 58)
print(gap_tbl.round(3).to_string())
print(f"""
  ▶ 선형 추세 (격차 ~ 연도)
     slope = {slope_h1:.4f} 명/천명/년
     R²    = {r2_h1:.4f}
     p     = {p_h1:.6f}  {'★ p < 0.001' if p_h1 < 0.001 else ''}

  ▶ 격차 변화
     2007년 : {gap_tbl.loc[2007,'격차']:.3f} 명/천명
     2023년 : {gap_tbl.loc[2023,'격차']:.3f} 명/천명
     배율   : {gap_tbl.loc[2023,'격차']/gap_tbl.loc[2007,'격차']:.1f}배

  ▶ 변이계수(CV) 추세
     slope = {sl_cv:.5f}/년   R² = {r_cv**2:.4f}   p = {p_cv:.6f}
     2007년 CV={cv_tbl.loc[2007,'CV']:.4f}  →  2023년 CV={cv_tbl.loc[2023,'CV']:.4f}
""")

## 셀 5. H2 통계 검정 — 고령화율 × 의사밀도 수도권 상호작용

In [ ]:
print("=" * 58)
print("  H2 검증: 고령화율 × 의사밀도  수도권 상호작용")
print("=" * 58)

# 집단별 피어슨 상관
for nm, sub in df_valid.groupby("수도권명"):
    r_, p_ = stats.pearsonr(sub["고령인구비율"], sub["의사밀도"])
    sl, *_ = stats.linregress(sub["고령인구비율"], sub["의사밀도"])
    print(f"  {nm:5s}  r={r_:+.4f}  p={p_:.4f}  slope={sl:.4f}  n={len(sub)}")

r_all, p_all = stats.pearsonr(df_valid["고령인구비율"], df_valid["의사밀도"])
print(f"  전체   r={r_all:+.4f}  p={p_all:.4f}  n={len(df_valid)}")

# 2024년 스냅샷
snap24 = df_valid[df_valid["연도"] == 2024].copy()
snap24["인구만"] = snap24["주민등록인구"] / 1_000_000
top_doc    = snap24.nlargest(1, "의사밀도").iloc[0]
bot_doc    = snap24.nsmallest(1, "의사밀도").iloc[0]
top3_age   = snap24.nlargest(3, "고령인구비율")[["지역","의사밀도","고령인구비율"]]

print(f"""
  ▶ 2024년 스냅샷
     의사밀도 최고 : {top_doc['지역']}  {top_doc['의사밀도']}명/천명  (고령 {top_doc['고령인구비율']}%)
     의사밀도 최저 : {bot_doc['지역']}  {bot_doc['의사밀도']}명/천명  (고령 {bot_doc['고령인구비율']}%)

  ▶ 고령인구비율 상위 3개 지역 (2024)""")
print(top3_age.to_string(index=False))

## 셀 6. H3 통계 검정 — 순이동 방향 × 의사밀도

In [ ]:
df_mv = df_valid.dropna(subset=["순이동인구", "유출입"])
유입   = df_mv[df_mv["유출입"] == "순유입"]["의사밀도"]
유출   = df_mv[df_mv["유출입"] == "순유출"]["의사밀도"]
t_h3, p_h3 = stats.ttest_ind(유입, 유출)
r_mv,  p_mv = stats.pearsonr(df_mv["순이동인구"], df_mv["의사밀도"])

print("=" * 58)
print("  H3 검증: 순이동 방향 × 의사밀도")
print("=" * 58)
print(f"""
  ▶ 집단 비교 (독립 t-검정)
     순유입 평균 = {유입.mean():.3f} 명/천명  (n={len(유입)})
     순유출 평균 = {유출.mean():.3f} 명/천명  (n={len(유출)})
     t = {t_h3:.3f}   p = {p_h3:.6f}  {'★ p < 0.001' if p_h3 < 0.001 else ''}

  ▶ 연속 상관 (피어슨)
     순이동인구 × 의사밀도  r = {r_mv:.4f}   p = {p_mv:.6f}
     {'→ 음의 상관: 유출이 많을수록 의사밀도 낮음' if r_mv < 0 else '→ 양의 상관'}
""")

## 셀 7. 그래프 1 — 수도권·비수도권 의사밀도 추세 + 격차 이중축

> **논문 활용**: 결과 섹션 H1 — 격차 심화 추세를 한눈에 보여주는 메인 그래프

In [ ]:
CLR_CAP  = "#2E75B6"
CLR_NCAP = "#C55A11"
CLR_GAP  = "#7030A0"
CLR_GRID = "#DDDDDD"

fig, ax1 = plt.subplots(figsize=(11, 5))

ax1.plot(gap_tbl.index, gap_tbl["수도권"],   color=CLR_CAP,  lw=2.5,
         marker="o", markersize=5, label="수도권")
ax1.plot(gap_tbl.index, gap_tbl["비수도권"], color=CLR_NCAP, lw=2.5,
         marker="s", markersize=5, label="비수도권")
ax1.fill_between(gap_tbl.index, gap_tbl["수도권"], gap_tbl["비수도권"],
                 alpha=0.12, color=CLR_GAP)
ax1.set_ylabel("인구 천 명당 의사 수 (명)", fontsize=11)
ax1.set_ylim(1.5, 5.5)
ax1.set_xlabel("연도", fontsize=11)
ax1.grid(axis="y", color=CLR_GRID, linewidth=0.8)

ax2 = ax1.twinx()
ax2.bar(gap_tbl.index, gap_tbl["격차"], color=CLR_GAP, alpha=0.35,
        width=0.5, label="격차 (우축)")
ax2.set_ylabel("수도권−비수도권 격차 (명)", fontsize=11, color=CLR_GAP)
ax2.tick_params(axis="y", colors=CLR_GAP)
ax2.set_ylim(0, 0.85)
ax2.annotate(
    f"slope = {slope_h1:.4f}/년\nR² = {r2_h1:.3f}  p < 0.001",
    xy=(2020, gap_tbl.loc[2020, "격차"]),
    xytext=(2013, 0.67),
    arrowprops=dict(arrowstyle="->", color=CLR_GAP, lw=1.2),
    color=CLR_GAP, fontsize=9,
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)
ax1.set_title(
    "그래프 1. 수도권·비수도권 의사밀도 추세 및 격차 변화 (2007~2024)",
    fontsize=12, fontweight="bold", pad=10,
)

fig.tight_layout()
fig.savefig("/content/results/plot_H1_gap_trend.png", bbox_inches="tight")
plt.show()
print("✅ 그래프 1 저장: /content/results/plot_H1_gap_trend.png")

## 셀 8. 그래프 2 — 변이계수(CV) 추세

> **논문 활용**: H1 보조 지표 — 특정 지역 쌍이 아닌 전체 시도 간 불평등이 심화됨을 보여줌

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(cv_tbl.index, cv_tbl["CV"], color=CLR_GAP, lw=2.3,
        marker="D", markersize=5, label="변이계수 CV")

x_fit = np.linspace(cv_tbl.index.min(), cv_tbl.index.max(), 200)
y_fit = sl_cv * x_fit + ic_cv
ax.plot(x_fit, y_fit, "--", color=CLR_GAP, lw=1.4, alpha=0.7, label="선형 추세")
ax.fill_between(cv_tbl.index, cv_tbl["CV"].min() * 0.97,
                cv_tbl["CV"], alpha=0.12, color=CLR_GAP)

ax.text(
    0.02, 0.92,
    f"slope={sl_cv:.5f}/년  R²={r_cv**2:.3f}  p={p_cv:.4f}",
    transform=ax.transAxes, fontsize=9, color=CLR_GAP,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=CLR_GAP, alpha=0.8),
)
ax.set_xlabel("연도", fontsize=11)
ax.set_ylabel("변이계수 (CV = 표준편차/평균)", fontsize=11)
ax.grid(axis="y", color=CLR_GRID, linewidth=0.8)
ax.legend(fontsize=9)
ax.set_title(
    "그래프 2. 시도별 의사밀도 지역 간 불평등 (변이계수) 추세 (2007~2024)",
    fontsize=12, fontweight="bold", pad=8,
)

fig.tight_layout()
fig.savefig("/content/results/plot_H1_CV_trend.png", bbox_inches="tight")
plt.show()
print("✅ 그래프 2 저장: /content/results/plot_H1_CV_trend.png")

## 셀 9. 그래프 3 — 고령화율 × 의사밀도 산점도 (수도권·비수도권 분리)

> **논문 활용**: H2 — 동일한 고령화 진행에도 수도권은 의사가 따라오고, 비수도권은 따라오지 않음을 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle(
    "그래프 3. 고령인구비율 × 의사밀도 산점도 — 수도권·비수도권 비교 (전 연도 패널)",
    fontsize=12, fontweight="bold",
)

for ax, (nm, sub), clr in zip(
    axes,
    df_valid.groupby("수도권명"),
    [CLR_NCAP, CLR_CAP],
):
    ax.scatter(sub["고령인구비율"], sub["의사밀도"],
               color=clr, s=22, alpha=0.55, edgecolors="none")
    sl, ic, r_, p_, _ = stats.linregress(sub["고령인구비율"], sub["의사밀도"])
    xs = np.linspace(sub["고령인구비율"].min(), sub["고령인구비율"].max(), 200)
    ax.plot(xs, sl * xs + ic, color=clr, lw=2,
            label=f"회귀선  r={r_:+.3f}  p={p_:.4f}")
    ax.set_title(nm, fontsize=12, color=clr, fontweight="bold")
    ax.set_xlabel("고령인구비율 (%)", fontsize=10)
    ax.set_ylabel("의사밀도 (명/천명)", fontsize=10)
    ax.grid(color=CLR_GRID, linewidth=0.7)
    ax.legend(fontsize=9)
    ax.text(0.03, 0.94, f"n={len(sub)}",
            transform=ax.transAxes, fontsize=9, color="gray")

fig.tight_layout()
fig.savefig("/content/results/plot_H2_scatter_aging_doctor.png", bbox_inches="tight")
plt.show()
print("✅ 그래프 3 저장: /content/results/plot_H2_scatter_aging_doctor.png")

## 셀 10. 그래프 4 — 2024년 버블차트 (고령화율 × 의사밀도, 버블=인구 규모)

> **논문 활용**: H2 보조 — 2024년 단면에서 수도권·비수도권의 위치가 얼마나 다른지 직관적으로 표현

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

for _, row in snap24.iterrows():
    clr  = CLR_CAP if row["수도권"] == 1 else CLR_NCAP
    size = row["인구만"] * 120 + 80
    ax.scatter(row["고령인구비율"], row["의사밀도"],
               s=size, color=clr, alpha=0.75, edgecolors="white", lw=0.8)
    short = (
        row["지역"]
        .replace("특별자치도", "").replace("특별자치시", "")
        .replace("광역시", "").replace("특별시", "")
        .replace("도", "")
    )
    ax.annotate(short, (row["고령인구비율"], row["의사밀도"]),
                textcoords="offset points", xytext=(6, 4),
                fontsize=8, color="#333333")

sl_all, ic_all, *_ = stats.linregress(snap24["고령인구비율"], snap24["의사밀도"])
xs = np.linspace(snap24["고령인구비율"].min(), snap24["고령인구비율"].max(), 200)
ax.plot(xs, sl_all * xs + ic_all, "--", color="gray",
        lw=1.5, alpha=0.7, label=f"전체 회귀선 (slope={sl_all:.3f})")

legend_elems = [
    Line2D([0],[0], marker="o", color="w", markerfacecolor=CLR_CAP,  markersize=10, label="수도권"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=CLR_NCAP, markersize=10, label="비수도권"),
    Line2D([0],[0], linestyle="--", color="gray", label="전체 회귀선"),
]
ax.legend(handles=legend_elems, fontsize=9, loc="upper left")
ax.set_xlabel("고령인구비율 (%, 2024)", fontsize=11)
ax.set_ylabel("인구 천 명당 의사 수 (명, 2024)", fontsize=11)
ax.grid(color=CLR_GRID, linewidth=0.8)
ax.set_title(
    "그래프 4. 2024년 시도별 고령인구비율 × 의사밀도 (버블 크기 = 인구 규모)",
    fontsize=12, fontweight="bold", pad=10,
)

fig.tight_layout()
fig.savefig("/content/results/plot_H2_bubble_2024.png", bbox_inches="tight")
plt.show()
print("✅ 그래프 4 저장: /content/results/plot_H2_bubble_2024.png")

## 셀 11. 그래프 5 — 순이동 방향별 의사밀도 박스플롯

> **논문 활용**: H3 — 인구 유출 지역과 유입 지역의 의사밀도 분포 차이를 통계 검정과 함께 제시

In [ ]:
df_box = df_mv[df_mv["유출입"].isin(["순유입", "순유출"])].copy()
order   = ["순유출", "순유입"]
palette = {"순유출": CLR_NCAP, "순유입": CLR_CAP}

fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(data=df_box, x="유출입", y="의사밀도", order=order,
            palette=palette, ax=ax, width=0.45, linewidth=0.9,
            flierprops=dict(marker="o", markersize=3, alpha=0.5))
sns.stripplot(data=df_box, x="유출입", y="의사밀도", order=order,
              palette=palette, ax=ax, size=3.5, alpha=0.45, jitter=True)

# 유의성 브래킷
y_top = df_box["의사밀도"].max() + 0.15
ax.plot([0, 0, 1, 1], [y_top - 0.05, y_top, y_top, y_top - 0.05],
        lw=1.2, color="black")
p_label = "p < 0.001" if p_h3 < 0.001 else f"p = {p_h3:.4f}"
ax.text(0.5, y_top + 0.03, f"★★★  {p_label}", ha="center", fontsize=10)

means = df_box.groupby("유출입")["의사밀도"].mean()
for i, grp in enumerate(order):
    ax.text(i, means[grp] + 0.08, f"μ={means[grp]:.3f}",
            ha="center", fontsize=9)

ax.set_xlabel("순이동 방향 (전 시도·전 연도)", fontsize=11)
ax.set_ylabel("인구 천 명당 의사 수 (명)", fontsize=11)
ax.grid(axis="y", color=CLR_GRID, linewidth=0.8)
ax.set_title("그래프 5. 순이동 방향별 의사밀도 분포 비교",
             fontsize=12, fontweight="bold", pad=8)

fig.tight_layout()
fig.savefig("/content/results/plot_H3_boxplot_migration.png", bbox_inches="tight")
plt.show()
print(f"✅ 그래프 5 저장: /content/results/plot_H3_boxplot_migration.png")
print(f"   순유출 n={len(df_box[df_box['유출입']=='순유출'])}  "
      f"순유입 n={len(df_box[df_box['유출입']=='순유입'])}  "
      f"t={t_h3:.3f}  p={p_h3:.6f}")

## 셀 12. 그래프 6 — 시도별 의사밀도 증가 속도 비교

> **논문 활용**: H1 보조 — 어느 지역이 빠르게 성장했고, 어느 지역이 뒤처졌는지 한눈에 비교

In [ ]:
pivot_doc    = df_valid.pivot_table(index="지역", columns="연도", values="의사밀도")
slope_region = {}
for region in pivot_doc.index:
    row = pivot_doc.loc[region].dropna()
    if len(row) >= 5:
        sl, *_ = stats.linregress(row.index.astype(float), row.values)
        slope_region[region] = sl

slope_df = pd.Series(slope_region).sort_values(ascending=False).reset_index()
slope_df.columns = ["지역", "slope"]
slope_df["수도권"] = slope_df["지역"].isin(CAPITAL)
slope_df["color"] = slope_df["수도권"].map({True: CLR_CAP, False: CLR_NCAP})

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(slope_df["지역"], slope_df["slope"],
               color=slope_df["color"], edgecolor="white", height=0.65)

for bar, val in zip(bars, slope_df["slope"]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=8.5)

ax.axvline(slope_df["slope"].mean(), color="gray", lw=1.2, ls="--")
legend_elems = [
    Line2D([0],[0], color=CLR_CAP,  lw=5, label="수도권"),
    Line2D([0],[0], color=CLR_NCAP, lw=5, label="비수도권"),
    Line2D([0],[0], color="gray",   lw=1.5, ls="--",
           label=f"전체 평균 ({slope_df['slope'].mean():.4f})"),
]
ax.legend(handles=legend_elems, fontsize=9, loc="lower right")
ax.set_xlabel("의사밀도 연간 증가율 (명/천명/년)", fontsize=11)
ax.grid(axis="x", color=CLR_GRID, linewidth=0.8)
ax.set_title(
    "그래프 6. 시도별 의사밀도 증가 속도 비교 (선형 추세 기울기, 2007~2024)",
    fontsize=12, fontweight="bold", pad=8,
)

fig.tight_layout()
fig.savefig("/content/results/plot_slope_by_region.png", bbox_inches="tight")
plt.show()
print("✅ 그래프 6 저장: /content/results/plot_slope_by_region.png")

## 셀 13. 그래프 7 — 2024년 고령화–의사밀도 사분면 분류

> **논문 활용**: H2 보조 — 의료 취약 지역('고령화↑ / 의사↓' 사분면)을 직접 식별

In [ ]:
x_med = snap24["고령인구비율"].median()
y_med = snap24["의사밀도"].median()

quad_labels = {
    (True,  True ): ("고령화↑ / 의사↑", "#D6E4F7"),
    (True,  False): ("고령화↑ / 의사↓ ★취약", "#FDDBD5"),
    (False, True ): ("고령화↓ / 의사↑", "#DFF5E3"),
    (False, False): ("고령화↓ / 의사↓", "#FFF3CD"),
}

fig, ax = plt.subplots(figsize=(10, 7))
ax.axhline(y_med, color="gray", lw=1.1, ls="--", alpha=0.7)
ax.axvline(x_med, color="gray", lw=1.1, ls="--", alpha=0.7)

for (high_age, high_doc), (label, bg) in quad_labels.items():
    x_range = (x_med if high_age else 0,  50 if high_age else x_med)
    y_range = (y_med if high_doc else 0,   8 if high_doc else y_med)
    ax.fill_betweenx([y_range[0], y_range[1]], x_range[0], x_range[1],
                     alpha=0.35, color=bg)
    ax.text(sum(x_range)/2, sum(y_range)/2, label,
            ha="center", va="center", fontsize=8.5, color="#555555",
            bbox=dict(fc="white", ec="none", alpha=0.6, pad=2))

for _, row in snap24.iterrows():
    clr = CLR_CAP if row["수도권"] == 1 else CLR_NCAP
    ax.scatter(row["고령인구비율"], row["의사밀도"],
               s=90, color=clr, alpha=0.85, edgecolors="white", lw=0.8, zorder=5)
    short = (
        row["지역"]
        .replace("특별자치도","").replace("특별자치시","")
        .replace("광역시","").replace("특별시","").replace("도","")
    )
    ax.annotate(short, (row["고령인구비율"], row["의사밀도"]),
                textcoords="offset points", xytext=(6, 3),
                fontsize=8, color="#222222", zorder=6)

legend_elems = [
    Line2D([0],[0], marker="o", color="w", markerfacecolor=CLR_CAP,  markersize=10, label="수도권"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor=CLR_NCAP, markersize=10, label="비수도권"),
]
ax.legend(handles=legend_elems, fontsize=9)
ax.set_xlabel("고령인구비율 (%, 2024)", fontsize=11)
ax.set_ylabel("인구 천 명당 의사 수 (명, 2024)", fontsize=11)
ax.grid(color=CLR_GRID, linewidth=0.6)
ax.set_title("그래프 7. 2024년 시도별 고령화–의사밀도 사분면 분류",
             fontsize=12, fontweight="bold", pad=8)

fig.tight_layout()
fig.savefig("/content/results/plot_quadrant_2024.png", bbox_inches="tight")
plt.show()
print("✅ 그래프 7 저장: /content/results/plot_quadrant_2024.png")

## 셀 14. 결과 CSV 저장

In [ ]:
import os

df_valid.to_csv("/content/results/panel_data_final.csv",
                index=False, encoding="utf-8-sig")
gap_tbl.round(4).to_csv("/content/results/H1_gap_table.csv",
                         encoding="utf-8-sig")
cv_tbl.round(4).to_csv("/content/results/H1_CV_table.csv",
                         encoding="utf-8-sig")
snap24[["지역","의사밀도","고령인구비율","주민등록인구","수도권명","유출입"]] \
    .sort_values("의사밀도", ascending=False) \
    .to_csv("/content/results/snapshot_2024.csv",
            index=False, encoding="utf-8-sig")
slope_df.to_csv("/content/results/slope_by_region.csv",
                index=False, encoding="utf-8-sig")

print("✅ CSV 저장 완료")
for f in sorted(os.listdir("/content/results")):
    sz = os.path.getsize(f"/content/results/{f}")
    print(f"   ✔ {f:<45s}  {sz:>8,} bytes")

## 셀 15. 논문 수치 요약 & ZIP 다운로드

결과 섹션 작성에 필요한 수치를 한 번에 출력합니다.

In [ ]:
print("=" * 62)
print("  📋 논문 결과 섹션 수치 요약 (복붙용)")
print("=" * 62)

print(f"""
[H1] 수도권·비수도권 의사밀도 격차 추세
  · 2007년 격차  : {gap_tbl.loc[2007,'격차']:.3f} 명/천명
  · 2023년 격차  : {gap_tbl.loc[2023,'격차']:.3f} 명/천명
  · 변화 배율    : {gap_tbl.loc[2023,'격차']/gap_tbl.loc[2007,'격차']:.1f}배
  · slope        : {slope_h1:.4f} 명/천명/년
  · R²           : {r2_h1:.4f}
  · p-value      : {p_h1:.6f}  (p < 0.001)
  · 변이계수 증가 : CV {cv_tbl.loc[2007,'CV']:.4f} → {cv_tbl.loc[2023,'CV']:.4f}  (p = {p_cv:.4f})

[H2] 고령화율 × 의사밀도 상호작용
  · 전체 패널    : r = {r_all:+.4f}  (p = {p_all:.4f})
""")

for nm, sub in df_valid.groupby("수도권명"):
    r_, p_ = stats.pearsonr(sub["고령인구비율"], sub["의사밀도"])
    print(f"  · {nm:5s}        : r = {r_:+.4f}  (p = {p_:.4f})")

print(f"""
  · 2024 최고    : {snap24.nlargest(1,'의사밀도').iloc[0]['지역']}  {snap24.nlargest(1,'의사밀도').iloc[0]['의사밀도']}명/천명
  · 2024 최저    : {snap24.nsmallest(1,'의사밀도').iloc[0]['지역']}  {snap24.nsmallest(1,'의사밀도').iloc[0]['의사밀도']}명/천명
  · 전라남도(고령 최고 지역) 의사밀도 : {snap24[snap24['지역']=='전라남도']['의사밀도'].values[0]}명/천명

[H3] 순이동 방향 × 의사밀도
  · 순유입 평균  : {유입.mean():.3f} 명/천명  (n={len(유입)})
  · 순유출 평균  : {유출.mean():.3f} 명/천명  (n={len(유출)})
  · t-검정       : t = {t_h3:.3f}  p = {p_h3:.6f}  (p < 0.001)
  · 피어슨 상관  : r = {r_mv:.4f}  (p = {p_mv:.6f})
""")

# ZIP 다운로드
import shutil
from google.colab import files

zip_path = "/content/results_hypothesis.zip"
shutil.make_archive("/content/results_hypothesis", "zip", "/content/results")
print(f"\n✅ 모든 분석 완료!")
print(f"   ZIP 파일 크기: {os.path.getsize(zip_path):,} bytes")
files.download(zip_path)